In [ ]:
#Liste Lesen

from pymarc import parse_xml_to_array

file_path = r"C:\Users\scaale00\Documents\GND-Python\data\authorities-gnd-kongress_dnbmarc_20250313.mrc.xml"

with open(file_path, "rb") as fh:
    records = parse_xml_to_array(fh)

for i, record in enumerate(records[:100]):
    print(f"\n--- Record {i + 1} ---")
    
    try:
        title = record.title or "(No title)"
        print(f"Title: {title}")
    except Exception as e:
        print(f"Title: [Error reading title] {e}")
    
    print(f"Leader: {record.leader}")

    try:
        for field in record.get_fields():
            print(field)
    except Exception as e:
        print(f"[Error printing fields] {e}")

In [ ]:
#GND Liste filtern (vif, level 1)

import csv
from pymarc import parse_xml_to_array

# File paths
file_path = r"C:\Users\scaale00\Documents\GND-Python\data\authorities-gnd-kongress_dnbmarc_20250313.mrc.xml"
output_csv = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records.csv"

# Load MARCXML records
with open(file_path, "rb") as fh:
    records = parse_xml_to_array(fh)

filtered_rows = []

for record in records:
    # ✅ Check for 075 $b == 'vif'
    is_vif = any(
        field.tag == "075" and 'vif' in field.get_subfields('b')
        for field in record.get_fields("075")
    )

    # ✅ Check for 042 $a in [gnd1, gnd2, gnd3]
    gnd_qualifiers = {"gnd1", "gnd2", "gnd3"}
    has_gnd_code = any(
        field.tag == "042" and
        any(code in gnd_qualifiers for code in field.get_subfields('a'))
        for field in record.get_fields("042")
    )

    if is_vif and has_gnd_code:
        gnd_id = record['001'].value() if record['001'] else ''

        # ✅ Extract Field 111 (Conference Name)
        field_111 = ""
        for field in record.get_fields("111"):
            subfields = field.get_subfields('a', 'b', 'c', 'd', 'e', 'n', 'q')  # relevant subfields
            if subfields:
                field_111 = " ".join(subfields)
                break  # only use the first 111 field

        # ✅ Extract Field 411 variants
        field_411_list = []
        for field in record.get_fields("411"):
            subfields = field.get_subfields('a', 'b', 'c', 'd', 'e', 'n', 'q')
            if subfields:
                name_variant = " ".join(subfields)
                field_411_list.append(name_variant.strip())

        field_411_joined = " | ".join(field_411_list)

        filtered_rows.append({
            'GND-ID': gnd_id,
            'Field 111': field_111,
            'Field 411': field_411_joined,
        })

# 💾 Write to CSV
with open(output_csv, mode='w', encoding='utf-8', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['GND-ID', 'Field 111', 'Field 411'])
    writer.writeheader()
    writer.writerows(filtered_rows)

print(f"✅ Exported {len(filtered_rows)} filtered records to: {output_csv}")

In [ ]:
#Varianten Aufräumen in der Basel Liste
 
import pandas as pd

# Load the Excel file
file_path = r"C:\Users\scaale00\Documents\GND-Python\data\VERZ_f-Level6-alle-MOCs_20240702_nme.xlsx"
df = pd.read_excel(file_path)

# Define the column name
column_name = "Abweichende Benennungen"

# Function to split and clean name variants
def split_name_variants(cell):
    if pd.isna(cell):
        return []
    return [name.strip() for name in str(cell).split(",") if name.strip()]

# Apply the cleaning function
df["Name Variants List"] = df[column_name].apply(split_name_variants)

# Preview
print(df[["Abweichende Benennungen", "Name Variants List"]].head())

# Optional: Save to a new Excel or CSV
output_path = r"C:\Users\scaale00\Documents\GND-Python\data\cleaned_name_variants.xlsx"
df.to_excel(output_path, index=False)

In [ ]:
#Varianten Aufräumen in der Liste GND

import pandas as pd
import re

# Load the CSV file
csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records.csv"
df = pd.read_csv(csv_path, dtype=str)  # Read as strings to avoid NaN issues

# Define the column name
column_name = "Field 411"

# Function to split and clean name variants from multiple delimiters
def split_variants(text):
    if pd.isna(text):
        return []
    # Split on both commas and pipes
    parts = re.split(r'[|,]', text)
    # Clean and deduplicate
    cleaned = {part.strip() for part in parts if part.strip()}
    return list(cleaned)

# Apply the function to create a clean list of variants
df["Field 411 Variants"] = df[column_name].apply(split_variants)

# Preview
print(df[[column_name, "Field 411 Variants"]].head())

# Optional: Save output
output_csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records_cleaned.csv"
df.to_csv(output_csv_path, index=False)

In [ ]:
#Code alle matches


import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import ast

# === File Paths ===
excel_path = r"C:\Users\scaale00\Documents\GND-Python\data\cleaned_name_variants.xlsx"
csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records_cleaned.csv"
output_path = r"C:\Users\scaale00\Documents\GND-Python\data\all_matches_with_variants.xlsx"

# === Load Excel and CSV ===
df_excel = pd.read_excel(excel_path)
df_csv = pd.read_csv(csv_path, dtype=str)

# === Parse Name Variants List (from string to list) ===
df_excel["Name Variants List"] = df_excel["Name Variants List"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

# === Normalize and Clean Fields ===
df_excel["title"] = df_excel["title"].fillna("").str.strip()
df_csv["Field 111"] = df_csv["Field 111"].fillna("").str.strip().str.lower()
df_csv["GND-ID"] = df_csv["GND-ID"].fillna("")

# === Split Field 411 into list of variants ===
df_csv["Field 411 Variants"] = df_csv["Field 411"].fillna("").apply(
    lambda x: [v.strip() for v in str(x).split("|") if len(v.strip()) >= 3]
)

# === Build Lookup Dictionaries ===
field_111_lookup = defaultdict(list)
variant_to_row_411 = defaultdict(list)

for _, row in df_csv.iterrows():
    field_111_lookup[row["Field 111"]].append(row)
    for variant in row["Field 411 Variants"]:
        v = variant.strip().lower()
        if len(v) >= 3:
            variant_to_row_411[v].append(row)

# === Matching Logic ===
results = []

for _, row in tqdm(df_excel.iterrows(), total=len(df_excel), desc="Matching entities"):
    title_orig = str(row["title"]).strip()
    title = title_orig.lower()
    datensatznummer = row.get("Datensatznummer", "")

    # Normalize variant list
    variants = [v.strip().lower() for v in row["Name Variants List"] if len(v.strip()) >= 3]
    matched_keys = set()

    # Match 1: title to Field 111
    if title in field_111_lookup:
        for csv_row in field_111_lookup[title]:
            key = (csv_row["Field 111"], csv_row["GND-ID"])
            if key not in matched_keys:
                results.append({
                    "Datensatznummer": datensatznummer,
                    "title": title_orig,
                    "Field 111": csv_row["Field 111"],
                    "GND-ID": csv_row["GND-ID"],
                    "match_type": "title"
                })
                matched_keys.add(key)

    # Match 2: variants to Field 111
    for variant in variants:
        if variant in field_111_lookup:
            for csv_row in field_111_lookup[variant]:
                key = (csv_row["Field 111"], csv_row["GND-ID"])
                if key not in matched_keys:
                    results.append({
                        "Datensatznummer": datensatznummer,
                        "title": title_orig,
                        "Field 111": csv_row["Field 111"],
                        "GND-ID": csv_row["GND-ID"],
                        "match_type": "variant_111"
                    })
                    matched_keys.add(key)

    # Match 3: variants to Field 411
    for variant in variants:
        if variant in variant_to_row_411:
            for csv_row in variant_to_row_411[variant]:
                key = (csv_row["Field 111"], csv_row["GND-ID"])
                if key not in matched_keys:
                    results.append({
                        "Datensatznummer": datensatznummer,
                        "title": title_orig,
                        "Field 111": csv_row["Field 111"],
                        "GND-ID": csv_row["GND-ID"],
                        "match_type": "variant_411"
                    })
                    matched_keys.add(key)

# === Save Output ===
results_df = pd.DataFrame(results)
results_df.to_excel(output_path, index=False)

print("✅ Matching complete. All matches saved to:", output_path)

In [ ]:
#Code für ein Match für Titel

import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import ast

# === File Paths ===
excel_path = r"C:\Users\scaale00\Documents\GND-Python\data\cleaned_name_variants.xlsx"
csv_path = r"C:\Users\scaale00\Documents\GND-Python\data\gnd_filtered_records_cleaned.csv"
output_path = r"C:\Users\scaale00\Documents\GND-Python\data\first_match_per_title.xlsx"

# === Load Excel and CSV ===
df_excel = pd.read_excel(excel_path)
df_csv = pd.read_csv(csv_path, dtype=str)

# === Parse Name Variants List (from string to list) ===
df_excel["Name Variants List"] = df_excel["Name Variants List"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

# === Normalize and Clean Fields ===
df_excel["title"] = df_excel["title"].fillna("").str.strip()
df_csv["Field 111"] = df_csv["Field 111"].fillna("").str.strip().str.lower()
df_csv["GND-ID"] = df_csv["GND-ID"].fillna("")

# === Split Field 411 into list of variants ===
df_csv["Field 411 Variants"] = df_csv["Field 411"].fillna("").apply(
    lambda x: [v.strip() for v in str(x).split("|") if len(v.strip()) >= 3]
)

# === Build Lookup Dictionaries ===
field_111_lookup = defaultdict(list)
variant_to_row_411 = defaultdict(list)

for _, row in df_csv.iterrows():
    field_111_lookup[row["Field 111"]].append(row)
    for variant in row["Field 411 Variants"]:
        v = variant.strip().lower()
        if len(v) >= 3:
            variant_to_row_411[v].append(row)

# === Matching Logic ===
results = []

for _, row in tqdm(df_excel.iterrows(), total=len(df_excel), desc="Matching one per entity"):
    title_orig = str(row["title"]).strip()
    title = title_orig.lower()
    datensatznummer = row.get("Datensatznummer", "")
    variants = [v.strip().lower() for v in row["Name Variants List"] if len(v.strip()) >= 3]

    match_found = False

    # Match 1: title to Field 111
    if not match_found and title in field_111_lookup:
        csv_row = field_111_lookup[title][0]
        results.append({
            "Datensatznummer": datensatznummer,
            "title": title_orig,
            "Field 111": csv_row["Field 111"],
            "GND-ID": csv_row["GND-ID"],
            "match_type": "title"
        })
        match_found = True

    # Match 2: variants to Field 111
    if not match_found:
        for variant in variants:
            if variant in field_111_lookup:
                csv_row = field_111_lookup[variant][0]
                results.append({
                    "Datensatznummer": datensatznummer,
                    "title": title_orig,
                    "Field 111": csv_row["Field 111"],
                    "GND-ID": csv_row["GND-ID"],
                    "match_type": "variant_111"
                })
                match_found = True
                break

    # Match 3: variants to Field 411
    if not match_found:
        for variant in variants:
            if variant in variant_to_row_411:
                csv_row = variant_to_row_411[variant][0]
                results.append({
                    "Datensatznummer": datensatznummer,
                    "title": title_orig,
                    "Field 111": csv_row["Field 111"],
                    "GND-ID": csv_row["GND-ID"],
                    "match_type": "variant_411"
                })
                match_found = True
                break

# === Save Output ===
results_df = pd.DataFrame(results)
results_df.to_excel(output_path, index=False)

print("✅ First match per title saved to:", output_path)